# Change detection: Sentinel-1 before and during the flood, scored on Copernicus EMS

[Notebook 01](01_flood_segmentation_s1.ipynb) found that one Sentinel-1 image is not enough: against EMS maps drawn on *other* satellites, a VV threshold and a per-pixel classifier both scored a CSI under 0.1. Dark, smooth surfaces are not all water: tarmac, dry bare fields and radar shadow look the same in a single image.

The standard remedy is to compare the flood pass with a pass **before** the flood from the **same relative orbit** (same viewing geometry): what became darker is new water. This notebook:

1. adds to every pair a pre-event reference (`copernicus-ems pairs --reference`): the latest pass of the same orbit before the activation's event time, on the same grid, within a budget of processing units;
2. builds a five-band dataset `VV, VH, VV_pre, VH_pre, dataMask`, split by area as before (`dataset --reference`);
3. compares, **on the same chips**, four detectors: the single-image threshold and classifier of notebook 01, and their change-detection counterparts;
4. scores them where EMS observed, apart for labels drawn on other satellites (independent) and on Sentinel-1 itself (same image).

It needs a free [Copernicus Data Space](https://dataspace.copernicus.eu) account (`CDSE_CLIENT_ID`, `CDSE_CLIENT_SECRET`). The event images of notebook 01 are reused; only the references are new (about 94 PU once).

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
RESOLUTION_M = 40
BUDGET_PU = float(os.environ.get("NOTEBOOK_BUDGET_PU", 100))  # nothing is sent above it
CHIP = 128
SEED = 0
LOOKBACK_DAYS = 36  # three repeat cycles of a relative orbit

FETCH = os.environ.get("NOTEBOOK_FETCH", "1") == "1"  # 0: use the pairs already in DATA_DIR
DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-change-s1.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. Pairs with a pre-event reference

`plan_pair` picks the flood pass, as in notebook 01. `plan_reference` then looks back from one day before the activation's `event_time` for the latest pass of the **same relative orbit** covering 90% of the AOI. Both are priced before anything is sent; images already in the data directory cost nothing.

In [ ]:
from datetime import timedelta

from copernicus_ems import RapidMapping
from copernicus_ems.sentinelhub import SentinelHub, make_pair, plan_pair, plan_reference

if FETCH:
    work = []
    async with RapidMapping(directory=DATA_DIR) as ems:
        started = {a.code: a.event_time for a in await ems.activations()}
        for code in ACTIVATIONS:
            before = started[code] - timedelta(days=1) if code in started else None
            for event in await ems.observed_events(code):
                if not (event.product.observed_time and event.product.aoi_extent):
                    continue
                plan = await plan_pair(ems, event, resolution_m=RESOLUTION_M)
                if plan.scene_pass is None:
                    continue
                ref = await plan_reference(
                    ems, plan, before=before, lookback=timedelta(days=LOOKBACK_DAYS)
                )
                work.append((plan, ref))
    to_fetch = {}
    for plan, ref in work:
        for path, pu in ((plan.backscatter_path, plan.pu), (ref.backscatter_path, ref.pu)):
            if path is not None and not path.is_file():
                to_fetch[path] = pu
    cost = sum(to_fetch.values())
    with_ref = sum(1 for _, ref in work if ref.scene_pass is not None)
    print(
        f"{len(work)} pairs, {with_ref} with a reference, {len(to_fetch)} images to fetch, {cost:.1f} PU"
    )
    assert cost <= BUDGET_PU, f"{cost:.1f} PU is over the budget of {BUDGET_PU}"
    async with SentinelHub() as hub:
        for plan, ref in work:
            await make_pair(hub, plan, reference=ref)
    print(f"spent about {hub.pu_estimated:.1f} PU")
    for plan, ref in work:
        if ref.scene_pass is None:
            print(
                f"  no reference for {plan.event.product.activation_code} {plan.product}: {ref.reason}"
            )
        else:
            lead = (plan.scene_pass.time - ref.scene_pass.time).total_seconds() / 86400
            print(
                f"  {plan.event.product.activation_code} {plan.product}: reference {lead:.0f} days before, orbit {ref.relative_orbit}"
            )

## 2. A five-band dataset, split by area

Only pairs with a reference are used. A pixel counts as data only where **both** passes have data; elsewhere the label becomes 255. Whole areas (activation + AOI) go to one split, as in notebook 01.

In [ ]:
import csv

from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
meta = json.loads((DATASET / "dataset.json").read_text())
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
print(f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px, bands {meta['image_bands']}")
for split, info in meta["splits"].items():
    print(
        f"  {split:5} {info['chips']:4d} chips  flooded share {info['flooded_share']}  areas: {', '.join(info['groups'])}"
    )

## 3. Features

Single-image features (as in notebook 01): VV and VH in dB, their ratio, local mean and spread of VV. Change features: the pre-event VV and VH, and the **differences** ΔVV = VV − VV_pre and ΔVH, raw and smoothed over 5×5 pixels. New open water shows as a strong drop (several dB) in both polarizations.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)  # the five features of notebook 01
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    )


def load(split):
    chosen = [r for r in rows if r["split"] == split]
    x = (
        np.stack([features(DATASET / r["image"]) for r in chosen])
        if chosen
        else np.empty((0, CHIP, CHIP, len(NAMES)))
    )
    y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in chosen]
    return chosen, x, y


data = {split: load(split) for split in ("train", "val", "test")}
for split, (chosen, _x, y) in data.items():
    observed = sum(int((label.labels != IGNORE).sum()) for label in y)
    wet = sum(int((label.labels == FLOODED).sum()) for label in y)
    print(
        f"{split:5} {len(chosen):4d} chips  {observed:9d} observed px  {wet / max(observed, 1):.1%} flooded"
    )

## 4. Four detectors

* **VV threshold** (notebook 01): water where VV < *t* dB.
* **Change threshold**: water where the flood pass is dark *and* got darker: VV < *t₁* dB and ΔVV (smoothed) < *t₂* dB; *t₁*, *t₂* chosen on the training areas for the best CSI.
* **Classifier, one image**: gradient-boosted trees on the five single-image features.
* **Classifier, change**: the same model on all eleven features.

The classifiers are trained on observed training pixels sampled one flooded to three dry, and their probability cut is chosen on the validation areas. Only labels 0 and 1 are used; 255 never is.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

from copernicus_ems.ml import Scores, evaluate


def pixels(split):
    _, x, y = data[split]
    labels = np.stack([label.labels for label in y])
    keep = labels != IGNORE
    return x[keep], (labels[keep] == FLOODED).astype(np.uint8)


def score(split, predict, keep=lambda _row: True):
    chosen, x, y = data[split]
    return sum(
        (evaluate(predict(x[i]), y[i]) for i in range(len(y)) if keep(chosen[i])), Scores.zero()
    )


best_t = max(
    np.arange(-30, -8, 0.25), key=lambda t: score("train", lambda f: f[..., 0] < t).csi or 0
)
pairs_t = [(t1, t2) for t1 in np.arange(-24, 0.5, 1.0) for t2 in np.arange(-8, 0.1, 0.5)]
best_t1, best_t2 = max(
    pairs_t,
    key=lambda t: score("train", lambda f: (f[..., 0] < t[0]) & (f[..., 9] < t[1])).csi or 0,
)
print(
    f"VV threshold: VV < {best_t:.2f} dB | change threshold: VV < {best_t1:.1f} dB and dVV < {best_t2:.1f} dB"
)

x_train, y_train = pixels("train")
rng = np.random.default_rng(SEED)
wet, dry = np.flatnonzero(y_train == 1), np.flatnonzero(y_train == 0)
n = min(len(wet), 100_000)
sample = np.concatenate(
    [rng.choice(wet, n, replace=False), rng.choice(dry, min(len(dry), 3 * n), replace=False)]
)


def fit(columns):
    model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
    model.fit(x_train[sample][:, columns], y_train[sample])

    def probability(f):
        return model.predict_proba(f[..., columns].reshape(-1, f[..., columns].shape[-1]))[
            :, 1
        ].reshape(f.shape[:-1])

    cut = max(
        np.arange(0.1, 0.95, 0.05),
        key=lambda p: score("val", lambda f: probability(f) > p).csi or 0,
    )
    return probability, cut


p_single, cut_single = fit(SINGLE)
p_change, cut_change = fit(slice(0, len(NAMES)))
print(
    f"classifiers trained on {len(sample):,} pixels; cuts p > {cut_single:.2f} (one image), p > {cut_change:.2f} (change)"
)

## 5. Scores on the test areas

Contingency counted **inside the observation masks only**; `unobserved km²` is water predicted outside every AOI, reported and never a false alarm. *independent*: EMS maps drawn on another satellite; *same image*: drawn on this Sentinel-1 pass.

In [ ]:
detectors = {
    "VV threshold": lambda f: f[..., 0] < best_t,
    "change threshold": lambda f: (f[..., 0] < best_t1) & (f[..., 9] < best_t2),
    "classifier 1 img": lambda f: p_single(f) > cut_single,
    "classifier change": lambda f: p_change(f) > cut_change,
}
subsets = {
    "all": lambda _row: True,
    "independent": lambda row: not row["same_image"],
    "same image": lambda row: row["same_image"],
}
test_rows = data["test"][0]
print("test chips:", {k: sum(1 for r in test_rows if keep(r)) for k, keep in subsets.items()})
print(
    f"{'':18} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'F1':>6} {'bias':>6} {'unobserved km²':>15}"
)
results = {}
for name, predict in detectors.items():
    for subset, keep in subsets.items():
        s = results[name, subset] = score("test", predict, keep)
        if s.hits + s.misses + s.false_alarms == 0:
            print(f"{name:18} {subset:12}   (no chip)")
            continue
        print(
            f"{name:18} {subset:12} {s.csi:6.3f} {s.pod:6.3f} {s.far:6.3f} {s.f1:6.3f} {s.bias:6.2f} {s.unobserved_predicted_km2:15.1f}"
        )

## 6. What the detectors see

Test chips with the most mapped flood, independent labels first: VV before and during the flood, ΔVV, the EMS label (blue flooded, beige dry, grey not observed), and the change-detection predictions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

chosen, x_test, y_test = data["test"]
order = sorted(
    range(len(y_test)),
    key=lambda i: (chosen[i]["same_image"], -int((y_test[i].labels == FLOODED).sum())),
)[:3]
label_colors = ListedColormap(["#e9dfc8", "#2b6cb0", "#bdbdbd"])

fig, axes = plt.subplots(len(order), 6, figsize=(17, 3.1 * len(order)), squeeze=False)
for row, i in enumerate(order):
    f = x_test[i]
    shown = np.where(y_test[i].labels == IGNORE, 2, y_test[i].labels)
    panels = [
        (f[..., 5], {"cmap": "gray", "vmin": -25, "vmax": 0}, "VV before (dB)"),
        (f[..., 0], {"cmap": "gray", "vmin": -25, "vmax": 0}, "VV during (dB)"),
        (f[..., 7], {"cmap": "RdBu", "vmin": -10, "vmax": 10}, "ΔVV (dB)"),
        (
            shown,
            {"cmap": label_colors, "vmin": 0, "vmax": 2, "interpolation": "nearest"},
            "EMS label",
        ),
        (
            detectors["change threshold"](f),
            {"cmap": "Blues", "vmin": 0, "vmax": 1},
            "change threshold",
        ),
        (
            detectors["classifier change"](f),
            {"cmap": "Blues", "vmin": 0, "vmax": 1},
            "classifier, change",
        ),
    ]
    for ax, (image, style, title) in zip(axes[row], panels, strict=True):
        ax.imshow(image, **style)
        ax.set_xticks([]), ax.set_yticks([])
        if row == 0:
            ax.set_title(title, fontsize=10)
    axes[row][0].set_ylabel(
        f"{chosen[i]['activation']} {chosen[i]['aoi']}\ngap {float(chosen[i]['gap_hours']):+.0f} h, map on\n{chosen[i]['made_from']}",
        fontsize=9,
    )
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=8,
)
fig.tight_layout(rect=(0, 0.02, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **Compare on the same chips.** The five-band dataset keeps only pixels where both passes have data, so its chips differ slightly from notebook 01's: the single-image rows above are the fair baseline, not notebook 01's table.
* **Independent labels are the test that matters.** Same-image labels share their pixels with the input; only maps drawn on other satellites say how a detector would do on a flood nobody has mapped yet.
* **What change detection cannot fix.** Water under vegetation or in towns brightens radar rather than darkening it (double bounce); a reference that is already wet (a flood that began before the activation's event time) hides the change; and an EMS map drawn a day away from the pass can show water that came or went in between (`gap_hours`).
* **A small dataset.** A few Italian activations, a few hundred chips: enough to compare detectors honestly, not to claim a general model.

The five-band dataset is the input a foundation model (Prithvi-EO-2.0, TerraMind via TerraTorch, with `ignore_index: 255`) would be fine-tuned on, scored with `copernicus_ems.ml.evaluate` as here.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*